# 常见的大模型优化方法：从KV Cache到Flash Attention


如图所示，在解码器的自回归生成过程中，给定一个输入，模型预测下一个词元，然后在下一步中将组合输入进行下一次预测。

![img](https://miro.medium.com/v2/resize:fit:1400/0*sexO6adGhaKr7aH0.gif)

这种自回归行为会重复一些操作，我们可以通过放大解码器中计算的掩码缩放点积注意力计算来更好地理解这一点。

![img](https://miro.medium.com/v2/resize:fit:1400/1*8xqD4AYTwn6mQXNw0uhDCg.gif)

由于解码器是因果的（即，一个标记（token）的注意力仅取决于其前面的标记），因此在每个生成步骤中，我们都在重新计算同一个先前标记的注意力，而我们实际上只想计算新标记的注意力。

# 1 注意力机制

标准Transformer的核心是缩放点积注意力（Scaled Dot-Product Attention）：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

对于长度为 $L$ 的序列，计算 $QK^T$ 会生成一个 $L \times L$ 的注意力分数矩阵。

- **时间复杂度**：$O(L^2)$
- **空间复杂度**：$O(L^2)$

随着上下文长度 $L$ 的增加（如从4k增加到128k），中间激活值的显存占用呈二次方爆炸式增长。
传统的Attention算子需要将庞大的注意力矩阵写入GPU的HBM（高带宽内存），然后再读回进行Softmax计算，
这种频繁的读写操作不仅耗时，且极易导致显存溢出（OOM）。

# 2. KV Cache

**KV Cache（键值缓存）** 是大模型推理中最基础的优化技术，其核心目的是以空间换时间，消除自回归生成中的冗余计算。

## 2.1 运行机制详解

在Transformer的每一层中，输入Token经过投影矩阵 $W_Q, W_K, W_V$ 变换为查询（Query）、键（Key）和值（Value）向量。

### 无缓存推理（Inefficient Decoding）

假设当前已生成序列为 "The sun rises"，需要预测下一个词。

1. 输入 "The sun rises"。
2. 模型重新计算 "The"、"sun"、"rises" 的所有 $Q, K, V$ 向量。
3. 计算注意力并输出 "in"。 这种方法的问题在于，"The"、"sun" 的 $K, V$ 向量在之前的步骤中已经计算过，且对于固定的历史上下文，它们是不会改变的 。

### 有缓存推理（Efficient Decoding with KV Cache）

1. **首Token生成（Prefill）**：计算Prompt中所有Token的 $K, V$，并将其存入显存中的特定区域（Cache）。
2. **后续生成（Decode）**：当生成第 $n$ 个Token时，仅需计算当前Token的 $q_n, k_n, v_n$。
3. **拼接与检索**：将 $k_n, v_n$ 追加到缓存末尾。注意力计算变为 $q_n$ 与 $K_{cache} + k_n$ 的交互。

通过缓存之前的键值对，我们可以专注于计算新token的注意力。

![img](https://miro.medium.com/v2/resize:fit:1400/1*uyuyOW1VBqmF5Gtv225XHQ.gif)

## 2.2 KV Cache 的显存挑战

虽然KV Cache将计算复杂度从 $O(L^2)$ 降低到了 $O(L)$（线性扫描历史），但它引入了巨大的显存开销。缓存的大小与以下因素成正比：

- 层数（Layers）
- 批处理大小（Batch Size）
- 序列长度（Sequence Length）
- 注意力头数（Heads）与维度（Head Dimension）

公式如下：

$$M_{KV} = 2 \times N_{layers} \times N_{heads} \times D_{head} \times L_{seq} \times B_{batch} \times P_{size}$$

随着Batch Size的增加（为了提高吞吐量）和Sequence Length的延长（长文档分析），KV Cache的体积往往会超过模型权重本身，成为限制并发量的主要瓶颈。这直接催生了注意力架构的演进。

# 3. 注意力架构的演进

为了缓解KV Cache带来的显存压力，研究界提出了一系列改进的注意力结构。这一演进过程本质上是在**模型表现力（Quality）**与**显存效率（Efficiency）**之间寻找最佳平衡点。

![img](https://camo.githubusercontent.com/96963fe92e1c743b7ede42513f411f36e7fd0f9ff12a6fa7108283a33628117d/68747470733a2f2f696d672d626c6f672e6373646e696d672e636e2f696d675f636f6e766572742f32663663626566383332333964616335633538383937643736346139646262332e776562703f782d6f73732d70726f636573733d696d6167652f666f726d61742c706e67)

## 3.1 Multi-Head Attention (MHA)：多头注意力

**定义**：MHA是《Attention Is All You Need》论文中提出的原始架构 。

- **结构**：拥有 $H$ 个Query头，$H$ 个Key头，和 $H$ 个Value头。
- **比例**：Query : Key : Value = $H : H : H$。
- **特点**：每个Query头都有其专属的Key和Value空间，能够独立地捕捉输入序列中不同的语义特征（如语法结构、指代关系等）。
- **缺点**：推理时显存占用极大。对于每一个Query头，都需要加载对应的Key/Value缓存，导致显存带宽压力巨大，严重限制了最大Batch Size。
- **代表模型**：GPT-3, Llama 1, Llama 2 (7B/13B)。

## 3.2 Multi-Query Attention (MQA)：多查询注意力

**定义**：MQA由Noam Shazeer在2019年提出 ，是一种极端的显存优化方案。

- **结构**：保留 $H$ 个Query头，但所有Query头**共享**同一个Key头和同一个Value头。
- **比例**：Query : Key : Value = $H : 1 : 1$。
- **显存优化**：KV Cache的大小直接缩小了 $H$ 倍。如果模型有32个头，KV Cache仅为MHA的1/32。
- **优势**：
  1. **极大提升吞吐量**：由于需要从HBM加载的数据量大幅减少，算术强度显著提高，使得GPU计算单元能被更充分利用。
  2. **支持超大Batch**：节省的显存可以用于增加Batch Size。
- **劣势**：
  1. **精度损失**：由于所有Query头只能在同一个Key-Value子空间中进行注意力检索，模型的表达能力受到限制，容易导致生成质量下降。
  2. **训练不稳定性**：早期实验表明MQA在训练初期容易出现不收敛的情况 。
- **代表模型**：Falcon-7B/40B, PaLM。

## 3.3 Grouped-Query Attention (GQA)：分组查询注意力

**定义**：GQA是Google Research在2023年提出的折中方案 ，现已成为主流LLM的标配。

- **结构**：将Query头分成 $G$ 个组（Group），每组包含 $H/G$ 个Query头。每个组共享一个Key头和一个Value头。
- **比例**：Query : Key : Value = $H : G : G$。
- **机制**：GQA可以看作是MHA和MQA的泛化形式。当 $G=H$ 时，它就是MHA；当 $G=1$ 时，它就是MQA。通常选取 $G=8$ 作为一个甜点（Sweet Spot）。
- **优势**：
  1. **精度接近MHA**：通过保留一定数量的KV头（如8个），模型仍能捕捉多样的语义特征，性能损失微乎其微。
  2. **速度接近MQA**：显存占用通常降至MHA的1/4或1/8（取决于分组数），大幅降低了显存带宽需求，允许更大的Batch Size。
  3. **平滑升级**：可以通过“Up-training”技术，将现有的MHA模型通过少量计算转换为GQA模型 。
- **代表模型**：Llama 2 (70B), Llama 3 (全系列), Mistral 7B, DeepSeek。

注意，GQA和MQA中，虽然KV的头数较少，但是Minimind会使用repeat_kv()函数来重复它们，再和Q相乘进行注意力计算。

举例：
| 类型 | Q Heads | KV Heads | \(W_Q\) | \(W_K/W_V\) |
|---|---:|---:|---:|---:|
| 单头 Attention | 1 | 1 | \(512\times512\) | \(512\times512\) |
| 8头 MHA | 8 | 8 | \(512\times512(8*64)\) | \(512\times512(**8***64)\) |
| 8头、2组 GQA | 8 | 2 | \(512\times512()8*64\) | \(512\times128(**2***64)\) |
| 8头 MQA | 8 | 1 | \(512\times512(8*64)\) | \(512\times64(**1***64)\) |

每种方法的总存储占用（Byte）= `batch_size × num_kv_heads × seq_len × head_dim × float_size`  


| 类型        | num_kv_heads | 计算公式                          | 显存占用（Byte） | 显存占用（MB） |
|-------------|---------------|-----------------------------------|------------------|----------------|
| **MHA**     | 32            | 1×32×1024×128×4 = 16,777,216      | 16,777,216       | **16 MB**      |
| **GQA (8组)** | 8             | 1×8×1024×128×4 = 4,194,304        | 4,194,304        | **4 MB**       |
| **MQA**     | 1             | 1×1×1024×128×4 = 524,288          | 524,288          | **0.5 MB**     |

# 4. Flash Attention：IO层面的优化

虽然GQA减少了KV Cache的**存储量**，但Flash Attention解决的是注意力计算过程中的**数据传输效率**问题。这是一种从GPU硬件特性出发的系统级优化。

Flash Attention是一种**显存优化 + 更快计算**的注意力实现，数学本质不变，只是在实现细节上做了大量工程优化，尤其适用于长序列Transformer模型（如LLM）。

它通过采用流式分块计算，在softmax前后都不存中间结果，避免显存瓶颈。提供更高吞吐、更长上下文能力而不会爆显存。

在PyTorch中，可以简单的通过`torch.nn.functional.scaled_dot_product_attention`直接使用Flash Attention。

以上介绍的KV cache，GQA，Flash Attention均已在MiniMind的Attention模块进行了实现，如下：

In [ ]:

def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    重复 Key/Value heads 以实现 GQA (Grouped Query Attention)
    
    GQA 是一种注意力机制优化，使用较少的 KV heads 来匹配更多的 Query heads。
    例如：8 个 Query heads 对应 2 个 KV heads，每个 KV head 需要重复 4 次。
    
    这样可以减少 KV 缓存的大小，在推理时节省显存。
    
    Args:
        x: Key 或 Value 张量 [batch, seq_len, num_kv_heads, head_dim]
        n_rep: 每个 KV head 需要重复的次数（n_rep = num_heads / num_kv_heads）
        也就是说，n_rep 个Qurey heads共享同一个KV head。
    Returns:
        重复后的张量 [batch, seq_len, num_heads, head_dim]
    """
    bs, slen, num_key_value_heads, head_dim = x.shape
    
    # 如果不需要重复（n_rep=1），直接返回
    if n_rep == 1:
        return x
    
    # 在维度 3 插入新维度，然后扩展并重塑
    # 例如：[B, L, 2, D] -> [B, L, 2, 1, D] -> [B, L, 2, 4, D] -> [B, L, 8, D]
    return (
        x[:, :, :, None, :]  # [B, L, num_kv_heads, 1, head_dim]
        .expand(bs, slen, num_key_value_heads, n_rep, head_dim)  # [B, L, num_kv_heads, n_rep, head_dim]
        .reshape(bs, slen, num_key_value_heads * n_rep, head_dim)  # [B, L, num_heads, head_dim]
    )

class Attention(nn.Module):
    """
    多头注意力机制（支持 GQA 和 Flash Attention）
    
    实现了标准的缩放点积注意力（Scaled Dot-Product Attention），支持：
    1. GQA (Grouped Query Attention): 使用较少的 KV heads 匹配更多的 Query heads
    2. Flash Attention: 使用 PyTorch 2.0+ 的优化注意力实现
    3. RoPE: 通过旋转位置编码将位置信息注入 Q 和 K
    4. KV Cache: 支持推理时的 KV 缓存加速
    
    注意力公式：
        Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
    """
    def __init__(self, args: MiniMindConfig):
        """
        初始化注意力层
        
        Args:
            args: MiniMindConfig 配置对象
        """
        super().__init__()
        # ========== GQA 配置 ==========
        # num_key_value_heads: KV heads 数量（通常小于 Query heads）
        self.num_key_value_heads = args.num_attention_heads if args.num_key_value_heads is None else args.num_key_value_heads
        # 确保 Query heads 数量能被 KV heads 数量整除
        assert args.num_attention_heads % self.num_key_value_heads == 0
        
        self.n_local_heads = args.num_attention_heads  # Query heads 数量
        self.n_local_kv_heads = self.num_key_value_heads  # KV heads 数量
        self.n_rep = self.n_local_heads // self.n_local_kv_heads  # 每个 KV head 需要重复的次数
        self.head_dim = args.hidden_size // args.num_attention_heads  # 每个头的维度
        
        # ========== 投影层 ==========
        # Q 投影：hidden_size -> num_heads * head_dim
        self.q_proj = nn.Linear(args.hidden_size, args.num_attention_heads * self.head_dim, bias=False)
        # K 投影：hidden_size -> num_kv_heads * head_dim（GQA：较少的 heads）
        self.k_proj = nn.Linear(args.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        # V 投影：hidden_size -> num_kv_heads * head_dim（GQA：较少的 heads）
        self.v_proj = nn.Linear(args.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        # 输出投影：num_heads * head_dim -> hidden_size
        self.o_proj = nn.Linear(args.num_attention_heads * self.head_dim, args.hidden_size, bias=False)
        
        # ========== Dropout ==========
        self.attn_dropout = nn.Dropout(args.dropout)  # 注意力分数 dropout
        self.resid_dropout = nn.Dropout(args.dropout)  # 残差连接 dropout
        self.dropout = args.dropout
        
        # ========== Flash Attention ==========
        # 检查是否支持 Flash Attention（需要 PyTorch >= 2.0）
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention') and args.flash_attn

    def forward(self,
                x: torch.Tensor,
                position_embeddings: Tuple[torch.Tensor, torch.Tensor],
                past_key_value: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                use_cache=False,
                attention_mask: Optional[torch.Tensor] = None):
        """
        前向传播
        
        Args:
            x: 输入张量 [batch, seq_len, hidden_size]
            position_embeddings: RoPE 位置编码 (cos, sin) 元组
            past_key_value: 缓存的 KV 值，用于增量解码 [batch, past_len, num_kv_heads, head_dim]
            use_cache: 是否返回 KV 缓存供下次使用
            attention_mask: 注意力掩码 [batch, seq_len]，1 表示有效位置，0 表示掩码位置
            
        Returns:
            output: 注意力输出 [batch, seq_len, hidden_size]
            past_kv: 新的 KV 缓存（如果 use_cache=True），否则为 None
        """
        bsz, seq_len, _ = x.shape
        
        # ========== 步骤 1：Q/K/V 投影，计算q，k，v ==========
        # 将输入投影到 Q、K、V 空间
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        
        # 拆分为多头格式：[batch, seq_len, num_heads, head_dim]
        # d_model = num_heads * head_dim
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)  # GQA：较少的 KV heads
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)  # GQA：较少的 KV heads
        
        # ========== 步骤 2：应用 RoPE 旋转位置编码 ==========
        # 将位置信息编码到 Q 和 K 中
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        
        # ========== 步骤 3：KV Cache 处理 ==========
        # 如果有缓存的 KV 值（增量解码），将其与当前 KV 拼接
        if past_key_value is not None:
            # past_key_value[0] 是缓存的 K，past_key_value[1] 是缓存的 V
            # 在序列维度（dim=1）上拼接：[batch, past_len+seq_len, num_kv_heads, head_dim]
            xk = torch.cat([past_key_value[0], xk], dim=1)
            xv = torch.cat([past_key_value[1], xv], dim=1)
        
        # 如果需要缓存，保存当前的 KV 值
        # 更新下一次的KV缓存
        past_kv = (xk, xv) if use_cache else None
        
        # ========== 步骤 4：GQA 处理 ==========
        # 调整维度顺序为 [batch, num_heads, seq_len, head_dim]（Flash Attention 格式）
        xq = xq.transpose(1, 2)  # [batch, num_heads, seq_len, head_dim]
        # 对于 KV，需要重复 heads 以匹配 Query heads 数量
        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)  # [batch, num_heads, kv_len, head_dim]
        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)  # [batch, num_heads, kv_len, head_dim]
        
        # ========== 步骤 5：计算注意力 ==========
        # 优先使用 Flash Attention（如果支持且条件满足）
        if self.flash and (seq_len > 1) and (past_key_value is None) and (attention_mask is None or torch.all(attention_mask == 1)):
            # Flash Attention：使用 PyTorch 优化的注意力实现
            # 条件：序列长度 > 1，没有 KV cache，没有复杂掩码
            output = F.scaled_dot_product_attention(
                xq, xk, xv,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True  # 自动应用因果掩码
            )
        else:
            # 标准注意力计算
            # 步骤 5.1：计算注意力分数 QK^T / sqrt(d_k)
            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)  # [batch, num_heads, seq_len, kv_len]
            
            # 步骤 5.2：应用因果掩码（只对当前序列部分）
            # 上三角矩阵掩码，防止看到未来的 token
            scores[:, :, :, -seq_len:] += torch.triu(
                torch.full((seq_len, seq_len), float("-inf"), device=scores.device),
                diagonal=1
            )
            
            # 步骤 5.3：应用注意力掩码（如果有）
            if attention_mask is not None:
                # 将掩码扩展到 [batch, 1, 1, seq_len] 并转换为分数掩码
                extended_attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
                extended_attention_mask = (1.0 - extended_attention_mask) * -1e9  # 0 -> -inf, 1 -> 0
                scores = scores + extended_attention_mask
            
            # 步骤 5.4：Softmax 归一化
            scores = F.softmax(scores.float(), dim=-1).type_as(xq)
            
            # 步骤 5.5：应用 dropout
            scores = self.attn_dropout(scores)
            
            # 步骤 5.6：加权求和
            output = scores @ xv  # [batch, num_heads, seq_len, head_dim]
        
        # ========== 步骤 6：输出投影 ==========
        # 重塑并投影回 hidden_size
        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)  # [batch, seq_len, num_heads * head_dim]
        output = self.resid_dropout(self.o_proj(output))  # [batch, seq_len, hidden_size]
        #o_proj 把多个 head 的结果进行可学习融合（没有改变拼接后的张量尺寸）
        
        return output, past_kv
    

# ⚠️
**每次 decode 生成新 token 时，本质上只是在计算“当前最新 token”的多层表示，让它通过 attention 读取上文；其他旧 token 不再更新，只提供之前缓存好的 K/V。**